## Install Dependencies

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

## Imports

In [ ]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

## Model Configuration

In [ ]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

## Load the Meeting Recording

In [ ]:
audio_filename = "content/denver_extract.mp3"
audio_file = open(audio_filename, "rb")

## Authenticate with Hugging Face

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Transcribe with Open-Source Whisper

In [ ]:
from transformers import pipeline 

pipe = pipeline(
    "automatic-speech-recognition",
    model = "openai/whisper-medium.en",
    dtype = torch.float16,
    device = 'cuda',
    return_timestamps = True
)

result= pipe(audio_filename)

transcription = result["text"]
open_source_transcription = transcription
print(transcription)

Device set to use cuda
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
 kind of the confluence of this whole idea of a Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So I'll let you see the creation of the logo here. Yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So, thank you. Thank you so much and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America, and to the republic for which it stands, one nation under God, indivisible, with liberty and justice for all. All right, thank you, Councilman Lopez, Madam Secretary, roll call. Black. Clark. Here. Espinosa. Here. Here Flynn Gilmore here Here cashman here can each Lopez yeah new Ortega here Sussman mr. President here 11 present 11 members present we do have a quorum approval the minutes are there any corrections to the minutes of October 2nd Seeing none minutes of October 2nd stand approve council announcements. Are there any announcements by members of council? Councilman Clark, thank you. Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway and lucky district 7 it will happen on Saturday, October 21st at 6 o'clock p.m It will move along Broadway from 3rd to Alameda It's gonna be a fun family-friendly event. Everyone's invited to come down wear a costume there will be candy for the kids and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on The fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween bread. Thank you Mr. President. All right. Thank you councilman Clark. I will be there. All right Presentations, Madam Secretary, do we have any presentations? None, Mr. President. Communications, do we have any communications? None, Mr. President. We do have one proclamation this evening, Proclamation 1127, an observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Lopez, will you please read it? Thank you, Mr. President, with pride. proclamation number 17, well let me just say this differently, proclamation number 1127 series of 2017 in observance of the second annual Indigenous Peoples Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the indigenous peoples have lived and flourished on the lands known as the America since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous indigenous tribes which include the Southern Ute, the Ute Mountain, Ute tribes of Colorado, and whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platte River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City.

system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

## Create the Meeting-Minutes Prompt

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

## Configure 4-Bit Quantization

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

## Load Llama and Generate the Minutes

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map = "auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 22 Aug 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
and kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue, so that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. Yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week as basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America and to the Republic for which it stands, one nation under God, indivisible, with liberty and justice for all. All right, thank you, Councilman Lopez. Madam Secretary, roll call. Black. Clark. Here. Espinosa. Here. Flynn. Gilmore. Here. Here. Cashman. Here. Kenich. Here. Lopez. Here. Nu. Here. Ortega. Here. Sussman. Here. Mr. President. Here. 11 present. 11 members present. We do have a quorum. Approval of the minutes. Are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd stand approved. Council announcements. Are there any announcements by members of council? Councilman Clark. Thank you, Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock p.m. It will move along Broadway from 3rd to Alameda. It's going to be a fun, family-friendly event. Everyone's invited to come down, wear a costume. There will be candy for the kids and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right, thank you, Councilman Clark. I will be there. All right. Presentations. Madam Secretary, do we have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening. Proclamation 1127 in observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Lopez, will you please read it? Thank you, Mr. President. Proclamation number 17, well, let me just say this differently. Proclamation number 1127 series of 2017 in observance of the second annual Indigenous Peoples Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the indigenous peoples have lived and flourished on the lands known as the Americas since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous indigenous tribes, which include the Southern Ute, the Ute Mountain Ute tribes of Colorado, and whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platt River Confluence gave bearing to the future settlements that would become the birthplace of the Mile High City.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Denver City Council Meeting Minutes**
**Date:** October 9th
**Location:** Denver City Council Chambers
**Attendees:**
- Council Members: Black, Clark, Espinosa, Flynn, Gilmore, Cashman, Kenich, Lopez, Nu, Ortega, Sussman
- Council President: Mr. President
- Invited Guests: Councilman Clark

**Summary:**
The Denver City Council meeting was held on Monday, October 9th, at the Denver City Council Chambers. Councilman Clark gave an announcement about the first-ever Halloween parade on Broadway in Lucky District 7, scheduled for October 21st. The meeting also featured a presentation on Indigenous Peoples Day and the proclamation of Proclamation 1127 in observance of the annual Indigenous Peoples Day in the City and County of Denver.

**Discussion Points:**

- The creation of the logo for Confluence Week, which represents the merging of two rivers and the importance of water.
- The significance of Indigenous Peoples Day and the recognition of the indigenous peoples' ancestral homelands in the City and County of Denver.
- The proclamation of Proclamation 1127 in observance of the annual Indigenous Peoples Day.

**Takeaways:**

- The importance of Indigenous Peoples Day and the recognition of the indigenous peoples' ancestral homelands in the City and County of Denver.
- The creation of the logo for Confluence Week, which represents the merging of two rivers and the importance of water.

**Action Items:**

- Councilman Clark: Organize the first-ever Halloween parade on Broadway in Lucky District 7 on October 21st.
- Council Members: Continue to recognize and celebrate Indigenous Peoples Day in the City and County of Denver.
- Council President: Continue to promote and support Confluence Week and Indigenous Peoples Day events.<|eot_id|>

## Decode the Model Output

In [ ]:
response = tokenizer.decode(outputs[0])
display(Markdown(response))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023 Today Date: 22 Aug 2026

You produce minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting. Please write minutes in markdown without code blocks, including:

a summary with attendees, location and date
discussion points
takeaways
action items with owners
Transcription: and kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue, so that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. Yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week as basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America and to the Republic for which it stands, one nation under God, indivisible, with liberty and justice for all. All right, thank you, Councilman Lopez. Madam Secretary, roll call. Black. Clark. Here. Espinosa. Here. Flynn. Gilmore. Here. Here. Cashman. Here. Kenich. Here. Lopez. Here. Nu. Here. Ortega. Here. Sussman. Here. Mr. President. Here. 11 present. 11 members present. We do have a quorum. Approval of the minutes. Are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd stand approved. Council announcements. Are there any announcements by members of council? Councilman Clark. Thank you, Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock p.m. It will move along Broadway from 3rd to Alameda. It's going to be a fun, family-friendly event. Everyone's invited to come down, wear a costume. There will be candy for the kids and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right, thank you, Councilman Clark. I will be there. All right. Presentations. Madam Secretary, do we have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening. Proclamation 1127 in observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Lopez, will you please read it? Thank you, Mr. President. Proclamation number 17, well, let me just say this differently. Proclamation number 1127 series of 2017 in observance of the second annual Indigenous Peoples Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the indigenous peoples have lived and flourished on the lands known as the Americas since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous indigenous tribes, which include the Southern Ute, the Ute Mountain Ute tribes of Colorado, and whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platt River Confluence gave bearing to the future settlements that would become the birthplace of the Mile High City.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Denver City Council Meeting Minutes Date: October 9th Location: Denver City Council Chambers Attendees:

Council Members: Black, Clark, Espinosa, Flynn, Gilmore, Cashman, Kenich, Lopez, Nu, Ortega, Sussman
Council President: Mr. President
Invited Guests: Councilman Clark
Summary: The Denver City Council meeting was held on Monday, October 9th, at the Denver City Council Chambers. Councilman Clark gave an announcement about the first-ever Halloween parade on Broadway in Lucky District 7, scheduled for October 21st. The meeting also featured a presentation on Indigenous Peoples Day and the proclamation of Proclamation 1127 in observance of the annual Indigenous Peoples Day in the City and County of Denver.

Discussion Points:

The creation of the logo for Confluence Week, which represents the merging of two rivers and the importance of water.
The significance of Indigenous Peoples Day and the recognition of the indigenous peoples' ancestral homelands in the City and County of Denver.
The proclamation of Proclamation 1127 in observance of the annual Indigenous Peoples Day.
Takeaways:

The importance of Indigenous Peoples Day and the recognition of the indigenous peoples' ancestral homelands in the City and County of Denver.
The creation of the logo for Confluence Week, which represents the merging of two rivers and the importance of water.
Action Items:

Councilman Clark: Organize the first-ever Halloween parade on Broadway in Lucky District 7 on October 21st.
Council Members: Continue to recognize and celebrate Indigenous Peoples Day in the City and County of Denver.
Council President: Continue to promote and support Confluence Week and Indigenous Peoples Day events.<|eot_id|>